# Stable-Baselines3 — State-of-the-Art RL Algorithms in 3 Lines

---

## What Is This Notebook About?

**Stable-Baselines3 (SB3)** is a library that gives you reliable, tested implementations of the best reinforcement learning algorithms — PPO, DQN, SAC, TD3, A2C — ready to use in 3 lines of code.

Think of it like **scikit-learn for RL**: just as `sklearn.ensemble.RandomForestClassifier` gives you a battle-tested Random Forest without implementing it yourself, SB3 gives you `PPO('MlpPolicy', env)` without implementing the algorithm from scratch.

By the end of this notebook you will understand:
- Why we need libraries like SB3 (RL is hard to implement correctly)
- PPO, DQN, SAC, A2C — what they are and when to use each
- How to train, evaluate, save, and load models
- How to customize policies and callbacks
- Hyperparameter tuning with Optuna
- A complete mini-project: training an agent to land a rocket

---

## Real-World Analogy: Hiring a Professional Trainer

Imagine you want to train an athlete. You could:
- **Option A**: Learn exercise science yourself, design every workout, track everything manually
- **Option B**: Hire a professional trainer who already knows all the best methods

Option A is implementing RL from scratch. Option B is Stable-Baselines3. The trainer (SB3) knows:
- Which exercises work (PPO, SAC, DQN)
- How to track progress (callbacks, logging)
- How to adjust difficulty (learning rate schedules)
- When to push harder and when to rest (exploration schedules)

You focus on defining the **problem** (the environment); SB3 handles the **training mechanics**.

---

## Why Does RL Need a Library Like SB3?

RL implementations are notoriously tricky. These bugs are all silent (no error, just bad results):
- Off-by-one in advantage estimation
- Wrong normalization of observations
- Forgetting to detach gradients when computing targets
- Wrong clipping in PPO's surrogate loss

SB3 has been tested extensively by the research community. Its implementations match the original papers.

---

## Prerequisites
- Python, NumPy basics
- Gymnasium notebook (environments, reset/step/reward loop)
- Basic neural network concepts (layers, gradients) helpful but not required

---

## Table of Contents
1. Installation & Setup
2. The RL Algorithm Zoo — Which One to Use?
3. PPO — The Most Popular Algorithm
4. DQN — Deep Q-Networks
5. SAC — For Continuous Control
6. Training, Evaluating, Saving & Loading
7. Callbacks — Monitor Training
8. Custom Policies
9. Common Pitfalls
10. Mini Project: LunarLander Training Pipeline
11. Interview Q&A
12. Resources

---

## Official Resources
- **Docs**: https://stable-baselines3.readthedocs.io/
- **GitHub**: https://github.com/DLR-RM/stable-baselines3
- **SB3 Zoo (pretrained models)**: https://github.com/DLR-RM/rl-baselines3-zoo
- **YouTube (PPO explained)**: https://www.youtube.com/watch?v=5P7I-xPq8u8
- **Hugging Face RL Course**: https://huggingface.co/learn/deep-rl-course/

## 1. Installation & Setup

In [ ]:
# Install commands:
# pip install stable-baselines3[extra]   # Includes tensorboard, optuna, etc.
# pip install gymnasium[classic-control,box2d]
# pip install optuna                      # For hyperparameter tuning

try:
    import stable_baselines3 as sb3
    from stable_baselines3 import PPO, DQN, SAC, A2C, TD3
    from stable_baselines3.common.env_util import make_vec_env
    from stable_baselines3.common.evaluation import evaluate_policy
    from stable_baselines3.common.callbacks import (
        EvalCallback, StopTrainingOnRewardThreshold, CheckpointCallback, CallbackList
    )
    from stable_baselines3.common.monitor import Monitor
    SB3_AVAILABLE = True
    print(f"Stable-Baselines3 version: {sb3.__version__}")
except ImportError:
    SB3_AVAILABLE = False
    print("SB3 not installed. Run: pip install stable-baselines3[extra]")
    print("All cells simulate output for learning purposes.")

try:
    import gymnasium as gym
    GYM_AVAILABLE = True
except ImportError:
    GYM_AVAILABLE = False

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')

BOTH_AVAILABLE = SB3_AVAILABLE and GYM_AVAILABLE
print(f"\nSB3: {'✓' if SB3_AVAILABLE else '✗'}  Gymnasium: {'✓' if GYM_AVAILABLE else '✗'}")
print("Setup complete!")

## 2. The RL Algorithm Zoo — Which One to Use?

Choosing the right algorithm is like choosing the right tool. A screwdriver works for screws; a hammer works for nails.

| Algorithm | Full Name | Action Space | Key Idea | Best For |
|-----------|-----------|--------------|----------|----------|
| **PPO** | Proximal Policy Optimization | Discrete & Continuous | Policy gradient with clipping | General purpose — start here |
| **DQN** | Deep Q-Network | Discrete only | Q-values via neural net | Atari, discrete games |
| **SAC** | Soft Actor-Critic | Continuous only | Maximum entropy RL | Robotics, physics sims |
| **TD3** | Twin Delayed DDPG | Continuous only | Two critics, delayed actor | Precise continuous control |
| **A2C** | Advantage Actor-Critic | Discrete & Continuous | Synchronous actor-critic | Faster than PPO, less stable |
| **HER** | Hindsight Experience Replay | With any | Learn from failures | Goal-conditioned tasks |

### Decision Tree:
```
What kind of actions?
├── Discrete (0,1,2,...n) ─────────► DQN or PPO
│     Simple game? → DQN
│     Complex/multi-env? → PPO
│
└── Continuous (real numbers) ─────► SAC or PPO
      Sample efficiency matters? → SAC (off-policy, more data efficient)
      Simplicity matters? → PPO (on-policy, easy to tune)
```

**When in doubt, start with PPO.** It works surprisingly well across a wide range of problems, is stable to train, and is easy to debug.

In [ ]:
# ── Algorithm Overview Visualization ─────────────────────────────────

algorithms = {
    'PPO': {
        'sample_efficiency': 5,  # 1-10 scale
        'stability': 9,
        'ease_of_use': 9,
        'discrete': True,
        'continuous': True,
        'color': '#3498db'
    },
    'DQN': {
        'sample_efficiency': 7,
        'stability': 7,
        'ease_of_use': 8,
        'discrete': True,
        'continuous': False,
        'color': '#e74c3c'
    },
    'SAC': {
        'sample_efficiency': 9,
        'stability': 8,
        'ease_of_use': 7,
        'discrete': False,
        'continuous': True,
        'color': '#2ecc71'
    },
    'TD3': {
        'sample_efficiency': 8,
        'stability': 7,
        'ease_of_use': 6,
        'discrete': False,
        'continuous': True,
        'color': '#f39c12'
    },
    'A2C': {
        'sample_efficiency': 4,
        'stability': 6,
        'ease_of_use': 8,
        'discrete': True,
        'continuous': True,
        'color': '#9b59b6'
    },
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Radar-like bar chart
metrics = ['Sample\nEfficiency', 'Training\nStability', 'Ease\nof Use']
x = np.arange(len(metrics))
width = 0.15
ax = axes[0]

for i, (name, data) in enumerate(algorithms.items()):
    vals = [data['sample_efficiency'], data['stability'], data['ease_of_use']]
    bars = ax.bar(x + i*width, vals, width, label=name, color=data['color'], alpha=0.8)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylabel('Score (1-10)')
ax.set_ylim(0, 11)
ax.set_title('Algorithm Comparison', fontweight='bold', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Action space compatibility
ax2 = axes[1]
algo_names = list(algorithms.keys())
discrete_support = [1 if algorithms[a]['discrete'] else 0 for a in algo_names]
continuous_support = [1 if algorithms[a]['continuous'] else 0 for a in algo_names]
colors = [algorithms[a]['color'] for a in algo_names]

y = np.arange(len(algo_names))
bars1 = ax2.barh(y - 0.2, discrete_support, 0.35, label='Discrete actions', color=colors, alpha=0.5, hatch='//')
bars2 = ax2.barh(y + 0.2, continuous_support, 0.35, label='Continuous actions', color=colors, alpha=0.9)

ax2.set_yticks(y)
ax2.set_yticklabels(algo_names, fontsize=12)
ax2.set_xlim(0, 1.5)
ax2.set_xticks([])
ax2.set_title('Action Space Compatibility', fontweight='bold', fontsize=12)
ax2.legend(fontsize=10)

# Add check/cross marks
for i, name in enumerate(algo_names):
    d = '✓' if algorithms[name]['discrete'] else '✗'
    c = '✓' if algorithms[name]['continuous'] else '✗'
    ax2.text(0.05, i - 0.2, f'  {d}', va='center', fontsize=14)
    ax2.text(0.05, i + 0.2, f'  {c}', va='center', fontsize=14)

plt.suptitle('Stable-Baselines3 Algorithm Guide', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/sb3_algorithms.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. PPO — The Most Popular RL Algorithm

**Proximal Policy Optimization (PPO)** is the workhorse of modern RL. It's used by:
- OpenAI to train ChatGPT's RLHF (Reinforcement Learning from Human Feedback)
- DeepMind for game-playing agents
- NASA for spacecraft control

### The Core Idea: Don't Change Too Much at Once

Traditional policy gradient methods (like REINFORCE) can take a huge update step that destroys the policy. PPO prevents this with a **clipping trick**:

```
L_CLIP = min(
    ratio × advantage,              ← Unconstrained update
    clip(ratio, 1-ε, 1+ε) × advantage  ← Clipped update
)

where ratio = π_new(a|s) / π_old(a|s)
      ε = 0.2 (typical clipping range)
```

If the new policy is too different from the old one (ratio > 1.2 or < 0.8), the gradient is clipped — the update is limited. This makes PPO stable.

### Key Hyperparameters:
| Parameter | Default | Meaning |
|-----------|---------|--------|
| `n_steps` | 2048 | Steps collected per update |
| `batch_size` | 64 | Mini-batch size for SGD |
| `n_epochs` | 10 | Number of SGD passes per update |
| `learning_rate` | 3e-4 | Adam learning rate |
| `clip_range` | 0.2 | PPO clipping ε |
| `gamma` | 0.99 | Discount factor |
| `gae_lambda` | 0.95 | GAE advantage estimation |

In [ ]:
# ── PPO: The Minimal Example ──────────────────────────────────────────

if BOTH_AVAILABLE:
    # The famous 3 lines of SB3
    env = gym.make('CartPole-v1')

    # Create model — 'MlpPolicy' = multi-layer perceptron (fully connected neural net)
    model = PPO(
        policy='MlpPolicy',
        env=env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        verbose=1   # Print training progress
    )

    print("Model architecture:")
    print(model.policy)

    # Train for 10k steps (tiny; real training: 100k-1M steps)
    model.learn(total_timesteps=10_000, progress_bar=False)

    # Evaluate the trained model
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10, deterministic=True)
    print(f"\nEvaluation: mean_reward={mean_reward:.1f} ± {std_reward:.1f}")
    print(f"Random baseline: ~21. Solved threshold: 475.")

    env.close()

else:
    print("=== PPO on CartPole-v1 ===")
    print()
    print("model = PPO('MlpPolicy', env, verbose=1)")
    print("model.learn(total_timesteps=10_000)")
    print()
    print("Model Architecture:")
    print("ActorCriticPolicy")
    print("  (pi_features_extractor): FlattenExtractor")
    print("  (mlp_extractor): MlpExtractor")
    print("    (policy_net): Sequential")
    print("      (0): Linear(4, 64)  ← 4 inputs = obs space")
    print("      (1): Tanh")
    print("      (2): Linear(64, 64)")
    print("      (3): Tanh")
    print("    (value_net): Sequential  ← Critic (same arch)")
    print("  (action_net): Linear(64, 2)  ← 2 outputs = actions")
    print("  (value_net): Linear(64, 1)   ← 1 output = value estimate")
    print()
    print("Training: -----------------------------------------")
    print("|   timesteps  |   reward  |   ep_len  |   loss  |")
    print("|   2048       |   22.4    |   22      |   0.029 |")
    print("|   4096       |   31.7    |   32      |   0.021 |")
    print("|   6144       |   55.3    |   55      |   0.015 |")
    print("|   8192       |   98.6    |   99      |   0.009 |")
    print("|   10240      |   142.1   |   142     |   0.006 |")
    print()
    print("Evaluation: mean_reward=142.1 ± 31.4")
    print("(More training needed for full solve; 10k steps is tiny)")

## 4. DQN — Deep Q-Networks

DQN is Q-Learning + a neural network. The neural network takes the **observation** as input and outputs **Q-values** for every possible action.

### What Makes DQN Work?

Two key innovations that stabilize training:

1. **Experience Replay Buffer**: Instead of learning from experiences one-by-one (which causes correlated updates), DQN stores all experiences `(s, a, r, s', done)` in a buffer and samples random batches. Like shuffling a deck of cards to break correlations.

2. **Target Network**: A frozen copy of the Q-network that's updated every N steps. Without it, we'd be chasing a moving target (the Q-target changes every update, making training unstable).

```
Main Q-Network (online) ─────► used to select actions
                          ─────► trained every step

Target Q-Network (frozen) ───► used to compute TD target
                          ─────► updated every C steps (e.g., every 1000 steps)
```

### DQN Update:
```
Target = r + γ × max_a'[Q_target(s', a')]   ← use frozen network
Loss = MSE(Q_online(s, a), Target)           ← train online network
```

In [ ]:
# ── DQN vs PPO: Key Differences ──────────────────────────────────────

# Conceptual comparison
comparison = {
    'Feature': ['Algorithm type', 'Data usage', 'Sample efficiency',
                'Action space', 'Memory needed', 'When to use'],
    'PPO': ['On-policy', 'Collected data used ONCE then discarded',
            'Lower (needs more data)', 'Discrete AND Continuous',
            'Low (no replay buffer)', 'Default choice, stable training'],
    'DQN': ['Off-policy', 'Stored in replay buffer, reused many times',
            'Higher (reuses old data)', 'Discrete ONLY',
            'High (buffer = 1M transitions)', 'Discrete actions, limited data'],
}

print("PPO vs DQN Comparison:")
print(f"{'Feature':<22} {'PPO':<40} {'DQN':<40}")
print("-" * 102)
for i in range(len(comparison['Feature'])):
    feat = comparison['Feature'][i]
    ppo_val = comparison['PPO'][i]
    dqn_val = comparison['DQN'][i]
    print(f"{feat:<22} {ppo_val:<40} {dqn_val:<40}")

print()
print("On-policy vs Off-policy explained:")
print("  On-policy (PPO): Can only learn from data collected by the CURRENT policy.")
print("    Like a student who only studies from their own notes.")
print("  Off-policy (DQN): Can learn from data collected by ANY policy (stored in buffer).")
print("    Like a student who can study from any notes ever written.")
print("  Off-policy = more sample efficient, but more complex to implement correctly.")

if BOTH_AVAILABLE:
    print("\n=== DQN on CartPole-v1 ===")
    env = gym.make('CartPole-v1')
    model = DQN(
        policy='MlpPolicy',
        env=env,
        learning_rate=1e-4,
        buffer_size=50_000,      # Replay buffer size
        learning_starts=1000,    # Start training after 1000 random steps
        batch_size=32,
        tau=1.0,                 # Target network update rate (1.0 = hard update)
        target_update_interval=500,  # Update target network every 500 steps
        exploration_fraction=0.1,    # Fraction of training with linear ε decay
        exploration_final_eps=0.05,  # Final ε value
        verbose=0
    )
    model.learn(total_timesteps=10_000)
    mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=10)
    print(f"DQN after 10k steps: mean_reward = {mean_r:.1f} ± {std_r:.1f}")
    env.close()
else:
    print("\n=== DQN on CartPole-v1 (simulated) ===")
    print("model = DQN('MlpPolicy', env, buffer_size=50_000, learning_starts=1000)")
    print("model.learn(total_timesteps=10_000)")
    print("DQN after 10k steps: mean_reward = 115.3 ± 28.7")

## 5. SAC — Soft Actor-Critic for Continuous Control

SAC is the go-to algorithm for **continuous action spaces** (robot joints, throttle/steering, etc.).

### The Key Innovation: Maximum Entropy RL

Normal RL: maximize cumulative reward.
SAC: maximize cumulative reward **PLUS entropy of the policy**.

```
SAC Objective = Σ E[r(s,a) + α × H(π(·|s))]
                           ↑
              Entropy bonus — reward the policy for being uncertain/diverse!
```

**Why add entropy?** A policy that's diverse (high entropy) keeps exploring. It finds multiple ways to solve a problem instead of committing to the first solution it finds. This makes SAC very robust and sample-efficient.

**α (temperature)** controls the trade-off. SAC automatically tunes α during training.

SAC also uses **two critic networks** (Twin Critics) to reduce overestimation bias — a common problem in Q-learning where the agent overestimates how good certain actions are.

In [ ]:
# ── SAC for Continuous Control ────────────────────────────────────────

if BOTH_AVAILABLE:
    try:
        # Pendulum-v1: swing a pendulum upright (continuous actions)
        env = gym.make('Pendulum-v1')

        model = SAC(
            policy='MlpPolicy',
            env=env,
            learning_rate=3e-4,
            buffer_size=100_000,
            batch_size=256,
            tau=0.005,           # Soft target network update
            gamma=0.99,
            ent_coef='auto',     # Auto-tune the temperature α
            verbose=0
        )
        model.learn(total_timesteps=10_000)
        mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=5)
        print(f"SAC on Pendulum-v1 (10k steps): mean_reward = {mean_r:.1f} ± {std_r:.1f}")
        print("(Full solve needs ~30k steps; perfect Pendulum score ≈ -200)")
        env.close()
    except Exception as e:
        print(f"Pendulum env issue: {e}")
        print("Try: pip install gymnasium[classic-control]")
else:
    print("=== SAC on Pendulum-v1 (simulated) ===")
    print("env = gym.make('Pendulum-v1')")
    print("model = SAC('MlpPolicy', env, ent_coef='auto')")
    print("model.learn(total_timesteps=10_000)")
    print()
    print("SAC on Pendulum-v1 (10k steps): mean_reward = -780.4 ± 120.3")
    print("(Full solve needs ~30k steps; perfect Pendulum score ≈ -200)")
    print()
    print("Pendulum-v1 observation: [cos(θ), sin(θ), θ_dot]")
    print("Pendulum-v1 action: torque in [-2.0, +2.0] (continuous!)")
    print()
    print("Why SAC for Pendulum?")
    print("  1. Action space is continuous → PPO or SAC (not DQN)")
    print("  2. SAC is more sample-efficient than PPO for this task")
    print("  3. ent_coef='auto' adapts the exploration automatically")

## 6. Training, Evaluating, Saving & Loading

A production RL workflow always involves these steps:
1. **Train** the model
2. **Evaluate** periodically to measure progress
3. **Save** the best model
4. **Load** for inference/deployment

In [ ]:
# ── Save, Load, and Evaluate Pattern ─────────────────────────────────

MODEL_SAVE_PATH = '/tmp/sb3_cartpole_ppo'
LOG_PATH = '/tmp/sb3_logs/'

if BOTH_AVAILABLE:
    # ── Train ────────────────────────────────────────────────────
    env = gym.make('CartPole-v1')
    model = PPO('MlpPolicy', env, verbose=0, tensorboard_log=LOG_PATH)
    model.learn(total_timesteps=20_000)

    # ── Evaluate ─────────────────────────────────────────────────
    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=20,    # Average over 20 episodes
        deterministic=True     # Use the best action (no exploration)
    )
    print(f"Trained model: {mean_reward:.1f} ± {std_reward:.1f}")

    # ── Save ─────────────────────────────────────────────────────
    model.save(MODEL_SAVE_PATH)
    print(f"\nModel saved to: {MODEL_SAVE_PATH}.zip")
    print(f"File size: {os.path.getsize(MODEL_SAVE_PATH + '.zip') / 1024:.1f} KB")

    # ── Load ─────────────────────────────────────────────────────
    loaded_model = PPO.load(MODEL_SAVE_PATH)
    print("\nModel loaded successfully!")

    # You must set the environment when loading for inference
    loaded_model.set_env(env)

    # Run one episode with the loaded model
    obs, _ = env.reset()
    total_reward = 0
    for _ in range(500):
        action, _states = loaded_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

    print(f"Loaded model episode reward: {total_reward:.0f}")
    env.close()

else:
    print("=== Save/Load Pattern (simulated) ===")
    print()
    print("# Train")
    print("model = PPO('MlpPolicy', env, verbose=0)")
    print("model.learn(total_timesteps=20_000)")
    print("Trained model: 312.4 ± 87.6")
    print()
    print("# Save")
    print("model.save('/tmp/my_ppo_model')")
    print("Saved to: /tmp/my_ppo_model.zip  (Size: 48.3 KB)")
    print()
    print("# Load")
    print("loaded = PPO.load('/tmp/my_ppo_model')")
    print("loaded.set_env(env)")
    print()
    print("# Inference")
    print("action, _ = loaded.predict(obs, deterministic=True)")
    print("  deterministic=True → always pick the best action (no random exploration)")
    print("  deterministic=False → sample from the policy distribution (evaluation matches training)")
    print()
    print("Loaded model episode reward: 312.0")

## 7. Callbacks — Monitor and Control Training

Callbacks are **hooks** that run during training at specific points. They let you:
- Stop training when the agent reaches a performance threshold
- Save checkpoints so you don't lose progress
- Log metrics for TensorBoard visualization
- Implement custom early stopping

Key built-in callbacks:
| Callback | Purpose |
|----------|---------|
| `EvalCallback` | Evaluate periodically, save the best model |
| `StopTrainingOnRewardThreshold` | Stop when reward exceeds threshold |
| `CheckpointCallback` | Save model every N steps |
| `CallbackList` | Chain multiple callbacks together |

In [ ]:
# ── Callback Training Pipeline ────────────────────────────────────────

if BOTH_AVAILABLE:
    os.makedirs('/tmp/sb3_best_model', exist_ok=True)
    os.makedirs('/tmp/sb3_checkpoints', exist_ok=True)

    # Separate env for evaluation (good practice — don't use training env)
    eval_env = gym.make('CartPole-v1')

    # Callback 1: Evaluate every 2000 steps, save the best model
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path='/tmp/sb3_best_model',
        log_path='/tmp/sb3_logs',
        eval_freq=2000,        # Evaluate every 2000 timesteps
        n_eval_episodes=5,     # Average over 5 episodes
        deterministic=True,
        verbose=1
    )

    # Callback 2: Stop if we solve CartPole (mean reward >= 450)
    stop_callback = StopTrainingOnRewardThreshold(
        reward_threshold=450,
        verbose=1
    )

    # Callback 3: Save checkpoint every 5000 steps
    checkpoint_callback = CheckpointCallback(
        save_freq=5000,
        save_path='/tmp/sb3_checkpoints',
        name_prefix='ppo_cartpole',
        verbose=0
    )

    # Chain them together
    callback = CallbackList([eval_callback, checkpoint_callback])
    # Note: stop_callback should be inside EvalCallback for proper behavior

    # Train with callbacks
    env = gym.make('CartPole-v1')
    model = PPO('MlpPolicy', env, verbose=0)
    model.learn(total_timesteps=30_000, callback=callback)

    print("\nTraining complete!")
    print(f"Best model saved at: /tmp/sb3_best_model/best_model.zip")

    eval_env.close()
    env.close()

else:
    print("=== Callback Pipeline (code explanation) ===")
    print()
    code = '''
# Evaluate every 2000 steps, save the best model automatically
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path='./best_model/',
    eval_freq=2000,
    n_eval_episodes=10,
    deterministic=True,
    verbose=1
)

# Save checkpoints every 5000 steps (so you don't lose progress!)
checkpoint_callback = CheckpointCallback(
    save_freq=5000,
    save_path='./checkpoints/',
    name_prefix='my_agent'
)

# Chain multiple callbacks
callback = CallbackList([eval_callback, checkpoint_callback])

# Train — callbacks fire automatically
model.learn(total_timesteps=100_000, callback=callback)

# Output during training:
# Eval at timestep 2000: mean_reward=54.3 (new best!)
# Eval at timestep 4000: mean_reward=112.7 (new best!)
# Checkpoint saved at 5000 steps
# Eval at timestep 6000: mean_reward=198.4 (new best!)
# ...
# Eval at timestep 50000: mean_reward=463.2 → STOPPING (threshold=450 reached!)
    '''
    print(code)

## 8. Vectorized Environments — Train Faster with Multiple Envs

Instead of running one environment and waiting for each step, you can run **N environments in parallel** and collect N experiences simultaneously. This is like having multiple training runs at once.

SB3 provides two types:
- `DummyVecEnv`: Runs envs sequentially in one process (easy but slower)
- `SubprocVecEnv`: Runs envs in separate processes (faster for slow envs)

**Speedup**: With 8 parallel envs, PPO collects data 8× faster. The total wall-clock time decreases significantly.

In [ ]:
# ── Vectorized Environments ───────────────────────────────────────────

if BOTH_AVAILABLE:
    import time

    # Single environment
    t0 = time.time()
    env_single = gym.make('CartPole-v1')
    model_single = PPO('MlpPolicy', env_single, verbose=0, n_steps=512)
    model_single.learn(total_timesteps=5_000)
    time_single = time.time() - t0
    env_single.close()

    # 4 parallel environments
    t0 = time.time()
    vec_env = make_vec_env('CartPole-v1', n_envs=4)  # 4 parallel envs
    model_multi = PPO('MlpPolicy', vec_env, verbose=0, n_steps=512)
    model_multi.learn(total_timesteps=5_000)
    time_multi = time.time() - t0
    vec_env.close()

    print(f"Single env:  {time_single:.2f}s for 5000 steps")
    print(f"4 envs:      {time_multi:.2f}s for 5000 steps")
    print(f"Speedup:     {time_single/time_multi:.2f}x")
    print("(Speedup is most dramatic for compute-heavy environments)")

else:
    print("=== Vectorized Environments ===")
    print()
    print("# Single env (baseline)")
    print("env = gym.make('CartPole-v1')")
    print("model = PPO('MlpPolicy', env)")
    print("model.learn(100_000)  # ~30 seconds")
    print()
    print("# 8 parallel envs (4x speedup!)")
    print("vec_env = make_vec_env('CartPole-v1', n_envs=8)")
    print("model = PPO('MlpPolicy', vec_env)")
    print("model.learn(100_000)  # ~8 seconds")
    print()
    print("How it works:")
    print("  Env 1: step() → obs1  ──┐")
    print("  Env 2: step() → obs2  ──┤")
    print("  Env 3: step() → obs3  ──┤─→ batch of 8 obs → PPO update")
    print("  ...                     ┤")
    print("  Env 8: step() → obs8  ──┘")
    print()
    print("All 8 environments step simultaneously → 8x more data per wall-clock second")

# Visualize the vectorized env concept
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

# Draw 8 environments
colors = plt.cm.tab10(np.linspace(0, 1, 8))
for i in range(8):
    x = 0.05 + i * 0.11
    rect = mpatches.FancyBboxPatch((x, 0.5), 0.08, 0.35,
                                    boxstyle='round,pad=0.01',
                                    facecolor=colors[i], edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 0.04, 0.67, f'Env\n{i+1}', ha='center', va='center', fontsize=9, fontweight='bold')
    # Arrow from env to agent
    ax.annotate('', xy=(0.5, 0.35), xytext=(x + 0.04, 0.50),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=1.5))

# Agent
agent_box = mpatches.FancyBboxPatch((0.38, 0.15), 0.24, 0.2,
                                     boxstyle='round,pad=0.02',
                                     facecolor='gold', edgecolor='black', linewidth=2)
ax.add_patch(agent_box)
ax.text(0.50, 0.25, 'PPO Agent\n(updates once per batch)', ha='center', va='center',
        fontsize=11, fontweight='bold')

ax.text(0.50, 0.92, 'Vectorized Environment: 8 Envs Running in Parallel',
        ha='center', va='center', fontsize=13, fontweight='bold')
ax.text(0.50, 0.05, 'Each env sends observations → Agent receives batch → One update',
        ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('/tmp/sb3_vecenv.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Common Pitfalls

In [ ]:
# ── Common SB3 Pitfalls ───────────────────────────────────────────────

print("=" * 68)
print(" Stable-Baselines3 Common Pitfalls")
print("=" * 68)

pitfalls = [
    {
        "title": "1. Using DQN for continuous action spaces",
        "symptom": "ValueError: DQN only supports Discrete action spaces",
        "fix": "Use PPO or SAC for continuous actions (Box action space)",
        "why": "DQN computes Q(s,a) for every action. With continuous spaces, \n       there are infinite actions — you can't enumerate them all."
    },
    {
        "title": "2. Not wrapping env with Monitor for reward tracking",
        "symptom": "evaluate_policy works, but EvalCallback has no log data",
        "fix": "env = Monitor(env, log_dir)  # Wrap before passing to algorithm",
        "why": "Monitor records episode rewards/lengths so callbacks can read them."
    },
    {
        "title": "3. Forgetting to set env when loading a saved model",
        "symptom": "AttributeError: 'NoneType' object has no attribute 'reset'",
        "fix": "model = PPO.load('path'); model.set_env(env)",
        "why": "Saved models don't store the environment. You must re-attach it."
    },
    {
        "title": "4. Not normalizing observations for continuous control",
        "symptom": "SAC/TD3 converges very slowly or not at all",
        "fix": "Use VecNormalize wrapper: env = VecNormalize(vec_env, norm_obs=True)",
        "why": "Neural networks work best with inputs near zero. Raw physics obs can be \n       huge (e.g., joint velocities in rad/s can be 100+). Normalize them!"
    },
    {
        "title": "5. Training for too few timesteps",
        "symptom": "Agent barely improves, looks random",
        "fix": "CartPole: 50-100k. LunarLander: 500k-1M. Atari: 10M+",
        "why": "RL needs lots of data. 10k steps is usually not enough. Start with \n       100k and increase if performance is still poor."
    },
    {
        "title": "6. Using deterministic=False during final evaluation",
        "symptom": "Evaluation scores vary wildly, hard to compare models",
        "fix": "evaluate_policy(model, env, deterministic=True)",
        "why": "deterministic=True always picks the best action (argmax). \n       deterministic=False adds noise for exploration — bad for evaluation."
    },
]

for p in pitfalls:
    print(f"\n{'─'*68}")
    print(f"  {p['title']}")
    print(f"  Symptom: {p['symptom']}")
    print(f"  Fix:     {p['fix']}")
    print(f"  Why:     {p['why']}")

print(f"\n{'='*68}")

## 10. Mini Project: LunarLander Training Dashboard

In [ ]:
# ── Mini Project: Train & Visualize LunarLander Agent ─────────────────
#
# LunarLander-v2: Land a rocket between two flags.
# Observation: [x, y, vx, vy, angle, angular_velocity, left_leg, right_leg]
# Actions: 0=do nothing, 1=fire left, 2=fire main, 3=fire right
# Solved: mean reward >= 200 over 100 episodes
#
# This project shows a complete pipeline:
# 1. Track rewards during training (not just after)
# 2. Compare against random baseline
# 3. Show sample efficiency curve

class RewardLogger:
    """Track rewards during training without callbacks."""
    def __init__(self):
        self.rewards = []
        self.episode_count = 0


def simulate_training_curve(n_episodes=200, algorithm='PPO', seed=42):
    """
    Simulate realistic training curves for LunarLander.
    PPO converges faster; A2C is noisier; DQN (not used here as action is Discrete) is efficient.
    """
    np.random.seed(seed)
    rewards = []
    # Three phases: random (-200 to -100), learning (-100 to 0), convergence (0 to 200+)
    for i in range(n_episodes):
        progress = i / n_episodes
        if algorithm == 'PPO':
            base = -200 + 400 * (1 - np.exp(-5 * progress))
            noise = 80 * np.exp(-3 * progress)
        elif algorithm == 'A2C':
            base = -200 + 370 * (1 - np.exp(-4 * progress))
            noise = 120 * np.exp(-2 * progress)
        elif algorithm == 'Random':
            base = -150
            noise = 50
        reward = base + np.random.normal(0, noise)
        rewards.append(max(-300, min(300, reward)))
    return rewards


# Simulate or train
if BOTH_AVAILABLE:
    try:
        print("Training PPO on LunarLander-v2 (200k steps)...")
        print("This may take 1-2 minutes. Set verbose=1 to see progress.")

        # Use vectorized envs for speed
        train_env = make_vec_env('LunarLander-v2', n_envs=4)
        model = PPO('MlpPolicy', train_env, verbose=0,
                    n_steps=1024, batch_size=64, n_epochs=4, gamma=0.999,
                    gae_lambda=0.98, ent_coef=0.01, learning_rate=1e-3)
        model.learn(total_timesteps=200_000)

        # Evaluate
        eval_env = gym.make('LunarLander-v2')
        mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
        print(f"\nFinal performance: {mean_r:.1f} ± {std_r:.1f}")
        print(f"Solved (>=200): {'YES!' if mean_r >= 200 else f'Not yet ({200 - mean_r:.0f} more points needed)'}")

        eval_env.close()
        train_env.close()

        # Use simulated curves for visualization
        ppo_rewards = simulate_training_curve(200, 'PPO')
        a2c_rewards = simulate_training_curve(200, 'A2C')
        random_rewards = simulate_training_curve(200, 'Random')
    except Exception as e:
        print(f"LunarLander not available: {e}")
        print("Try: pip install gymnasium[box2d]")
        ppo_rewards = simulate_training_curve(200, 'PPO')
        a2c_rewards = simulate_training_curve(200, 'A2C')
        random_rewards = simulate_training_curve(200, 'Random')
else:
    ppo_rewards = simulate_training_curve(200, 'PPO')
    a2c_rewards = simulate_training_curve(200, 'A2C')
    random_rewards = simulate_training_curve(200, 'Random')


# ── Visualization Dashboard ───────────────────────────────────────────
window = 20

def smooth(arr, w):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('LunarLander-v2 Training Dashboard: PPO vs A2C vs Random',
             fontsize=14, fontweight='bold')

# 1. Training curves
ax = axes[0, 0]
episodes = range(len(ppo_rewards))
ax.plot(episodes, ppo_rewards, alpha=0.15, color='blue')
ax.plot(range(window-1, len(ppo_rewards)), smooth(ppo_rewards, window), 'b-', linewidth=2.5, label='PPO')
ax.plot(episodes, a2c_rewards, alpha=0.15, color='green')
ax.plot(range(window-1, len(a2c_rewards)), smooth(a2c_rewards, window), 'g-', linewidth=2.5, label='A2C')
ax.plot(episodes, random_rewards, alpha=0.15, color='red')
ax.plot(range(window-1, len(random_rewards)), smooth(random_rewards, window), 'r--', linewidth=1.5, label='Random')
ax.axhline(200, color='gold', linestyle='--', linewidth=2, label='Solved (200)')
ax.set_title('Training Curves', fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Distribution at end of training
ax = axes[0, 1]
late_ppo = ppo_rewards[-50:]
late_a2c = a2c_rewards[-50:]
late_random = random_rewards[-50:]
ax.hist(late_ppo, bins=15, alpha=0.7, color='blue', label=f'PPO (mean={np.mean(late_ppo):.0f})')
ax.hist(late_a2c, bins=15, alpha=0.7, color='green', label=f'A2C (mean={np.mean(late_a2c):.0f})')
ax.hist(late_random, bins=10, alpha=0.7, color='red', label=f'Random (mean={np.mean(late_random):.0f})')
ax.axvline(200, color='gold', linestyle='--', linewidth=2, label='Solved (200)')
ax.set_title('Score Distribution (Last 50 Episodes)', fontweight='bold')
ax.set_xlabel('Reward')
ax.set_ylabel('Count')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 3. Learning rate: how quickly does % of episodes > 0 increase?
ax = axes[1, 0]
for rewards, name, color in [(ppo_rewards, 'PPO', 'blue'), (a2c_rewards, 'A2C', 'green')]:
    positive_rate = [np.mean(np.array(rewards[max(0,i-window):i+1]) > 0) * 100
                     for i in range(len(rewards))]
    ax.plot(positive_rate, color=color, linewidth=2, label=name)
ax.set_title('% Episodes with Positive Reward (rolling)', fontweight='bold')
ax.set_xlabel('Episode')
ax.set_ylabel('% Positive Reward')
ax.set_ylim(0, 105)
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Hyperparameter sensitivity table
ax = axes[1, 1]
ax.axis('off')
table_data = [
    ['Hyperparameter', 'PPO Default', 'Effect if Too High', 'Effect if Too Low'],
    ['learning_rate', '3e-4', 'Unstable, diverges', 'Learns too slowly'],
    ['n_steps', '2048', 'Slower updates', 'High variance estimates'],
    ['clip_range', '0.2', 'Policy changes too much', 'Learns too slowly'],
    ['gamma', '0.99', 'Agent plans too far ahead', 'Short-sighted behavior'],
    ['gae_lambda', '0.95', 'High variance', 'High bias in advantage'],
    ['ent_coef', '0.0', '—', 'Premature convergence'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                  cellLoc='center', loc='center',
                  colWidths=[0.2, 0.15, 0.32, 0.33])
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.5)

# Header styling
for j in range(4):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

ax.set_title('PPO Hyperparameter Guide', fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('/tmp/sb3_lunarlander_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nLunarLander Training Summary:")
print(f"  PPO final mean reward:    {np.mean(ppo_rewards[-20:]):.1f}")
print(f"  A2C final mean reward:    {np.mean(a2c_rewards[-20:]):.1f}")
print(f"  Random baseline:          {np.mean(random_rewards):.1f}")
print(f"  PPO episodes to cross 0:  {next((i for i,r in enumerate(smooth(ppo_rewards,10)) if r > 0), 200)}")

## 11. Interview Q&A

---

### Q1: What is the difference between on-policy and off-policy algorithms?
**A**: On-policy (PPO, A2C): The data used for training must come from the **current policy**. After each update, the old data is discarded. Off-policy (DQN, SAC, TD3): Can learn from data collected by any previous policy, stored in a replay buffer. Off-policy = more sample efficient (reuses data) but more complex. On-policy = simpler, more stable, needs more data.

---

### Q2: How does PPO prevent catastrophic policy updates?
**A**: PPO uses a clipped surrogate objective. It computes the probability ratio `r(t) = π_new(a|s) / π_old(a|s)`. If this ratio is too far from 1 (policy changed too much), the gradient is clipped. The objective becomes `min(r(t) × advantage, clip(r(t), 1-ε, 1+ε) × advantage)`. This means if the new policy would make a large change, the gradient signal is cut off, preventing destructive updates.

---

### Q3: What is the replay buffer in DQN and why is it needed?
**A**: The replay buffer stores past experiences `(state, action, reward, next_state, done)`. Without it, training on sequential experience has two problems: (1) **Correlation**: consecutive experiences are highly correlated (the cart is in similar positions for many steps), leading to biased updates. (2) **Forgetting**: without reuse, rare but important experiences are learned from only once. The replay buffer breaks correlation by sampling random batches, and allows reusing rare experiences many times.

---

### Q4: When would you choose SAC over PPO?
**A**: Choose SAC when: (1) Actions are continuous (robot control, physics simulation), (2) Sample efficiency matters (you have limited environment interaction budget), (3) You want automatic entropy tuning. Choose PPO when: (1) Actions might be discrete or continuous, (2) Simplicity and stability are more important than sample efficiency, (3) You're new to RL and want reliable defaults.

---

### Q5: What is the advantage function and why does PPO use GAE?
**A**: The advantage `A(s,a) = Q(s,a) - V(s)` measures "how much better is action a compared to the average action in state s?". PPO uses GAE (Generalized Advantage Estimation) to balance bias vs variance in advantage estimation. `A_GAE = Σ (γλ)^t δ_{t+k}` where δ is the TD error. `λ=1` gives high variance (Monte Carlo), `λ=0` gives high bias (TD). `λ=0.95` is the sweet spot.

---

### Q6: How does RLHF (Reinforcement Learning from Human Feedback) work for ChatGPT?
**A**: Three steps: (1) **Supervised fine-tuning (SFT)**: Fine-tune base LLM on high-quality human demonstrations. (2) **Reward model training**: Collect human preferences ("which response is better?"), train a reward model to predict human preference scores. (3) **PPO fine-tuning**: Use PPO to optimize the LLM's policy against the learned reward model, subject to a KL divergence constraint that keeps it close to the SFT model. The result: an LLM that generates responses humans prefer, without needing explicit programming of "good response" rules.

## 12. Resources

### Official
- **SB3 Docs**: https://stable-baselines3.readthedocs.io/en/master/
- **SB3 GitHub**: https://github.com/DLR-RM/stable-baselines3
- **RL Baselines3 Zoo** (hyperparameter tuning recipes): https://github.com/DLR-RM/rl-baselines3-zoo
- **SB3 Contrib** (experimental algorithms): https://github.com/Stable-Baselines-Team/stable-baselines3-contrib

### Tutorials & Courses
- **Hugging Face Deep RL Course (FREE)**: https://huggingface.co/learn/deep-rl-course/
- **Nicholas Renotte Full Course**: https://www.youtube.com/watch?v=Mut_u40Sqz4
- **PPO paper explained (Yannic Kilcher)**: https://www.youtube.com/watch?v=5P7I-xPq8u8

### Papers
- **PPO**: https://arxiv.org/abs/1707.06347
- **DQN**: https://arxiv.org/abs/1312.5602
- **SAC**: https://arxiv.org/abs/1801.01290
- **TD3**: https://arxiv.org/abs/1802.09477
- **RLHF (InstructGPT)**: https://arxiv.org/abs/2203.02155

---

## Summary

| Concept | Takeaway |
|---------|----------|
| Algorithm choice | Discrete → PPO/DQN; Continuous → SAC/PPO; Default → PPO |
| On-policy | Learn from current policy data only (PPO, A2C) |
| Off-policy | Learn from any past data in replay buffer (DQN, SAC) |
| PPO key idea | Clip policy updates to prevent catastrophic changes |
| DQN key idea | Q-network + replay buffer + target network |
| SAC key idea | Maximize reward + entropy (explore and exploit) |
| Callbacks | Hook into training to save best model, stop early |
| VecEnv | Run N envs in parallel for N× faster data collection |

**Next**: RLlib — scale RL to clusters with hundreds of parallel workers!